# Modelado MLP (2018–2024)

Este notebook forma parte del pipeline de ciencia de datos del proyecto **crash-severity-predictor**.

El objetivo es desarrollar, entrenar y evaluar un modelo **Multilayer Perceptron (MLP)** para la predicción de severidad en hechos de tránsito a partir del dataset procesado durante las fases de EDA y ETL. Esta implementación constituye la tercera iteración dentro del conjunto de modelos candidatos del proyecto.


In [1]:
# -- Importaciones ----------------------------------------------
import pandas as pd
import numpy as np
from sklearn.neural_network import MLPClassifier
from sklearn.preprocessing import StandardScaler
from sklearn.utils.class_weight import compute_sample_weight

from sklearn.metrics import (
    classification_report,
    confusion_matrix,
    roc_auc_score,
    f1_score,
    accuracy_score,
    roc_curve,
    precision_score,
    recall_score
)

import plotly.graph_objects as go
import plotly.express as px
import json, os
import warnings
warnings.filterwarnings('ignore')

print('✓ Librerías cargadas correctamente')

✓ Librerías cargadas correctamente


## 1. Carga de datos

In [2]:
# -- Carga ----------------------------------------------
train = pd.read_parquet('../data/clean/train.parquet')
test  = pd.read_parquet('../data/clean/test.parquet')

FEATURES = ['tipo_eve','tipo_veh','g_hora_5','dia_sem_ocu',
            'sexo_per','edad_quinquenales','mayor_menor','depto_ocu']
TARGET = 'fall_les'

X_train = train[FEATURES]
y_train = train[TARGET]
X_test  = test[FEATURES]
y_test  = test[TARGET]

scaler = StandardScaler()
X_train_sc = scaler.fit_transform(X_train)
X_test_sc  = scaler.transform(X_test)

print(f'Train : {X_train_sc.shape}')
print(f'Test  : {X_test_sc.shape}')
print(f'\n✓ Features escaladas correctamente')

Train : (57553, 8)
Test  : (14389, 8)

✓ Features escaladas correctamente


## 2. Entrenamiento MLP

In [3]:
# -- Entrenamiento con sample_weight para manejar desbalance ----------------------------------------------
sample_weights = compute_sample_weight(class_weight='balanced', y=y_train)

mlp = MLPClassifier(
    hidden_layer_sizes=(128, 64, 32),
    activation='relu',
    solver='adam',
    learning_rate_init=0.001,
    max_iter=100,
    early_stopping=True,
    validation_fraction=0.1,
    random_state=42,
    verbose=False
)

mlp.fit(X_train_sc, y_train, sample_weight=sample_weights)
print(f'✓ Modelo entrenado correctamente')
print(f'Capas ocultas  : {mlp.hidden_layer_sizes}')
print(f'Iteraciones    : {mlp.n_iter_}')
print(f'Loss final     : {mlp.loss_:.4f}')

✓ Modelo entrenado correctamente
Capas ocultas  : (128, 64, 32)
Iteraciones    : 27
Loss final     : 0.6052


## 3. Evaluación del modelo

In [4]:
# -- Predicciones ----------------------------------------------
y_pred = mlp.predict(X_test_sc)

# Probabilidad de la clase "Fallecido" (clase 1)
y_proba = mlp.predict_proba(X_test_sc)[:, 0]

# Convertir y_test a binario:
# 1 = Fallecido
# 0 = Lesionado
y_test_bin = (y_test == 1).astype(int)

# Métricas
acc = accuracy_score(y_test, y_pred)
f1 = f1_score(y_test, y_pred, average='weighted')
auc = roc_auc_score(y_test_bin, y_proba)
cm = confusion_matrix(y_test, y_pred)

print(f'Clases del modelo: {mlp.classes_}')

print('\n=== MLP ===')
print(f'Accuracy : {acc:.4f}')
print(f'F1-Score : {f1:.4f}')
print(f'ROC-AUC  : {auc:.4f}')

print()
print(classification_report(
    y_test,
    y_pred,
    target_names=['Fallecido', 'Lesionado']
))

print('Matriz de confusión:')
print(cm)

Clases del modelo: [1 2]

=== MLP ===
Accuracy : 0.6526
F1-Score : 0.6889
ROC-AUC  : 0.7162

              precision    recall  f1-score   support

   Fallecido       0.31      0.68      0.43      2748
   Lesionado       0.89      0.65      0.75     11641

    accuracy                           0.65     14389
   macro avg       0.60      0.66      0.59     14389
weighted avg       0.78      0.65      0.69     14389

Matriz de confusión:
[[1859  889]
 [4110 7531]]


In [5]:
# -- Visualización 1: Métricas ----------------------------------------------
fig_metricas = go.Figure(go.Bar(
    x=['Accuracy', 'F1-Score', 'ROC-AUC'],
    y=[acc, f1, auc],
    text=[f'{acc:.4f}', f'{f1:.4f}', f'{auc:.4f}'],
    textposition='auto',
    marker_color=['#8338EC', '#3A86FF', '#FF006E'],
    width=0.4
))
fig_metricas.update_layout(
    title='Métricas de evaluación — MLP',
    yaxis=dict(range=[0, 1], title='Valor'),
    xaxis_title='Métrica',
    height=400,
    template='plotly_white'
)
fig_metricas.show()

In [6]:
# -- Visualización 2: Matriz de confusión ----------------------------------------------
fig_cm = px.imshow(
    cm,
    labels=dict(x='Predicción', y='Real', color='Cantidad'),
    x=['Fallecido', 'Lesionado'],
    y=['Fallecido', 'Lesionado'],
    text_auto=True,
    color_continuous_scale='Purples',
    title='Matriz de confusión — MLP'
)
fig_cm.update_layout(height=400, template='plotly_white')
fig_cm.show()

In [7]:
# -- Visualización 3: Curva ROC ----------------------------------------------
fpr, tpr, _ = roc_curve(y_test_bin, y_proba)

fig_roc = go.Figure()
fig_roc.add_trace(go.Scatter(
    x=fpr,
    y=tpr,
    mode='lines',
    name=f'MLP (AUC = {auc:.4f})',
    line=dict(color='#8338EC', width=2.5)
))

fig_roc.add_trace(go.Scatter(
    x=[0,1],
    y=[0,1],
    mode='lines',
    name='Baseline (AUC = 0.5)',
    line=dict(color='gray', width=1.5, dash='dash')
))

fig_roc.update_layout(
    title='Curva ROC — MLP',
    xaxis_title='Tasa de Falsos Positivos',
    yaxis_title='Tasa de Verdaderos Positivos',
    height=450,
    template='plotly_white',
    legend=dict(x=0.6, y=0.1)
)

fig_roc.show()

In [8]:
# -- Visualización 4: Curva de pérdida ----------------------------------------------
fig_loss = go.Figure()
fig_loss.add_trace(go.Scatter(
    y=mlp.loss_curve_,
    mode='lines',
    name='Loss entrenamiento',
    line=dict(color='#8338EC', width=2.5)
))
fig_loss.add_trace(go.Scatter(
    y=mlp.validation_scores_,
    mode='lines',
    name='Score validación',
    line=dict(color='#FF006E', width=2.5, dash='dot')
))
fig_loss.update_layout(
    title='Curva de pérdida — MLP',
    xaxis_title='Iteración',
    yaxis_title='Valor',
    height=420,
    template='plotly_white',
    legend=dict(x=0.6, y=0.9)
)
fig_loss.show()

In [9]:
# -- Guardar modelo y scaler ----------------------------------------------
import joblib
import os

os.makedirs('../data/models', exist_ok=True)
joblib.dump(mlp, '../data/models/mlp.pkl')
joblib.dump(scaler, '../data/models/scaler_mlp.pkl')

resultados_mlp = {
    'modelo'             : 'MLP',
    'accuracy'           : round(acc, 4),
    'f1_score'           : round(f1, 4),
    'roc_auc'            : round(auc, 4),
    'precision_fallecido': round(precision_score(y_test, y_pred, pos_label=1), 4),
    'recall_fallecido'   : round(recall_score(y_test, y_pred, pos_label=1), 4),
    'iteraciones'        : mlp.n_iter_,
    'loss_final'         : round(mlp.loss_, 4),
}

with open('../data/models/resultados_mlp.json', 'w') as f:
    json.dump(resultados_mlp, f, indent=2)

print('✓ Modelo guardado en data/models/mlp.pkl')
print('✓ Scaler guardado en data/models/scaler_mlp.pkl')
print(f'\nResumen MLP:')
for k, v in resultados_mlp.items():
    print(f'  {k:<25} {v}')

✓ Modelo guardado en data/models/mlp.pkl
✓ Scaler guardado en data/models/scaler_mlp.pkl

Resumen MLP:
  modelo                    MLP
  accuracy                  0.6526
  f1_score                  0.6889
  roc_auc                   0.7162
  precision_fallecido       0.3114
  recall_fallecido          0.6765
  iteraciones               27
  loss_final                0.6052


## 4. Resumen del modelo

| Métrica | Valor |
|---|---|
| Accuracy | 65.26% |
| F1-Score (weighted) | 68.89% |
| ROC-AUC | 71.62% |
| Precision Fallecido | 31% |
| Recall Fallecido | 67% |
| Iteraciones | 27 |
| Loss final | 0.6052 |

**Conclusiones:**
- sample_weight="balanced" corrigió el Recall de 9% a 68%
- Convergencia en 27 iteraciones con early stopping
- Loss desciende consistentemente — entrenamiento estable
- Score de validación oscila levemente ; comportamiento normal con desbalance

## 5. Split de validación para selección de hiperparámetros

`X_train_final` y `X_val` ya no se recalculan aquí: se cargan desde `train_final.parquet` y `val.parquet` (sin escalar), generados una sola vez en `05_dataset_modeling.ipynb` (mismo `train_test_split(test_size=0.25, random_state=42, stratify=y_train)`). `X_test` no se toca en ningún momento de esta sección. Para la búsqueda se usa un `StandardScaler` ajustado **solo** con `X_train_final` (no con `X_val`).

In [10]:
# -- Cargar split de validación centralizado (generado en 05_dataset_modeling.ipynb) ----------------------------------------------
from sklearn.model_selection import ParameterSampler

train_final = pd.read_parquet('../data/clean/train_final.parquet')
val = pd.read_parquet('../data/clean/val.parquet')

X_train_final = train_final[FEATURES]
y_train_final = train_final[TARGET]
X_val = val[FEATURES]
y_val = val[TARGET]

total_modelado = len(X_train) + len(X_test)
print(f'Total dataset de modelado (train+test) : {total_modelado:,}')
print(f'X_train_final : {X_train_final.shape[0]:,}  ({X_train_final.shape[0]/total_modelado*100:.2f}% del total)')
print(f'X_val         : {X_val.shape[0]:,}  ({X_val.shape[0]/total_modelado*100:.2f}% del total)')
print(f'X_test        : {X_test.shape[0]:,}  ({X_test.shape[0]/total_modelado*100:.2f}% del total)')

# Escalado propio para la búsqueda: el scaler se ajusta SOLO con X_train_final
scaler_search = StandardScaler()
X_train_final_sc = scaler_search.fit_transform(X_train_final)
X_val_sc = scaler_search.transform(X_val)

Total dataset de modelado (train+test) : 71,942
X_train_final : 43,164  (60.00% del total)
X_val         : 14,389  (20.00% del total)
X_test        : 14,389  (20.00% del total)


## 6. Búsqueda de hiperparámetros (25 combinaciones aleatorias, evaluadas en X_val)

`early_stopping=True` usa su propio 10% interno tomado de `X_train_final` (vía `validation_fraction=0.1`), no de `X_val`.

In [11]:
# -- Búsqueda aleatoria manual: entrena SOLO con X_train_final, evalúa F1 weighted SOLO con X_val ----------------------------------------------
sample_weights_final = compute_sample_weight(class_weight='balanced', y=y_train_final)

param_grid_mlp = {
    'hidden_layer_sizes': [(64,32), (128,64,32), (100,50), (128,64)],
    'learning_rate_init': [0.0005, 0.001, 0.005],
    'alpha'             : [0.0001, 0.001, 0.01],
}
sampler_mlp = list(ParameterSampler(param_grid_mlp, n_iter=25, random_state=42))

resultados_busqueda_mlp = []
for params in sampler_mlp:
    modelo_tmp = MLPClassifier(
        **params, activation='relu', solver='adam', max_iter=100,
        early_stopping=True, validation_fraction=0.1, random_state=42
    )
    modelo_tmp.fit(X_train_final_sc, y_train_final, sample_weight=sample_weights_final)
    pred_val = modelo_tmp.predict(X_val_sc)
    f1_val = f1_score(y_val, pred_val, average='weighted')
    resultados_busqueda_mlp.append({**params, 'f1_val': f1_val, 'n_iter': modelo_tmp.n_iter_})

df_busqueda_mlp = pd.DataFrame(resultados_busqueda_mlp).sort_values('f1_val', ascending=False).reset_index(drop=True)
print(df_busqueda_mlp.to_string())

mejor_mlp = max(resultados_busqueda_mlp, key=lambda d: d['f1_val'])
mejores_params_mlp = {k: v for k, v in mejor_mlp.items() if k not in ('f1_val', 'n_iter')}
print(f'\nMejor F1 (validación): {mejor_mlp["f1_val"]:.4f}')
print(f'Mejores hiperparámetros (MLP): {mejores_params_mlp}')

    learning_rate_init hidden_layer_sizes   alpha    f1_val  n_iter
0               0.0010           (64, 32)  0.0001  0.708476      39
1               0.0050          (100, 50)  0.0100  0.701983      31
2               0.0050          (100, 50)  0.0001  0.695784      44
3               0.0005          (128, 64)  0.0010  0.695480      51
4               0.0010           (64, 32)  0.0010  0.695169      34
5               0.0005           (64, 32)  0.0010  0.694105      47
6               0.0005           (64, 32)  0.0100  0.693893      46
7               0.0005           (64, 32)  0.0001  0.693629      47
8               0.0050          (128, 64)  0.0001  0.684282      19
9               0.0050           (64, 32)  0.0001  0.681664      27
10              0.0005          (100, 50)  0.0100  0.680751      27
11              0.0010      (128, 64, 32)  0.0001  0.677261      45
12              0.0005          (128, 64)  0.0001  0.672792      43
13              0.0050      (128, 64, 32)  0.010

## 7. Reentrenamiento final (train_final + val) y evaluación en test

In [12]:
# -- Reentrenar UNA sola vez con los mejores hiperparámetros sobre X_train completo, evaluar UNA sola vez en X_test ----------------------------------------------
scaler_v2 = StandardScaler()
X_train_sc_v2 = scaler_v2.fit_transform(X_train)
X_test_sc_v2  = scaler_v2.transform(X_test)
sample_weights_v2 = compute_sample_weight(class_weight='balanced', y=y_train)

mlp_v2 = MLPClassifier(
    **mejores_params_mlp, activation='relu', solver='adam', max_iter=100,
    early_stopping=True, validation_fraction=0.1, random_state=42
)
mlp_v2.fit(X_train_sc_v2, y_train, sample_weight=sample_weights_v2)

pred_train_v2 = mlp_v2.predict(X_train_sc_v2)
pred_test_v2  = mlp_v2.predict(X_test_sc_v2)
proba_test_v2 = mlp_v2.predict_proba(X_test_sc_v2)[:, list(mlp_v2.classes_).index(1)]  # 1 = Fallecido (Deceased)
y_test_bin_v2 = (y_test == 1).astype(int)

acc_train_v2 = accuracy_score(y_train, pred_train_v2)
acc_test_v2  = accuracy_score(y_test, pred_test_v2)
f1_test_v2   = f1_score(y_test, pred_test_v2, average='weighted')
auc_test_v2  = roc_auc_score(y_test_bin_v2, proba_test_v2)
prec_fall_v2 = precision_score(y_test_bin_v2, (pred_test_v2 == 1).astype(int))
rec_fall_v2  = recall_score(y_test_bin_v2, (pred_test_v2 == 1).astype(int))
overfit_v2   = acc_train_v2 - acc_test_v2

print('=== MLP v2 (hiperparámetros seleccionados con validación) ===')
print(f'Hiperparámetros      : {mejores_params_mlp}')
print(f'Iteraciones          : {mlp_v2.n_iter_}')
print(f'Accuracy train       : {acc_train_v2:.4f}')
print(f'Accuracy test        : {acc_test_v2:.4f}')
print(f'F1 weighted (test)   : {f1_test_v2:.4f}')
print(f'ROC-AUC (test)       : {auc_test_v2:.4f}')
print(f'Precision Deceased   : {prec_fall_v2:.4f}')
print(f'Recall Deceased      : {rec_fall_v2:.4f}')
print(f'Overfitting (train-test acc) : {overfit_v2:.4f}')

joblib.dump(mlp_v2, '../data/models/mlp_v2.pkl')
joblib.dump(scaler_v2, '../data/models/scaler_mlp_v2.pkl')
print('\n✓ Modelo guardado en data/models/mlp_v2.pkl (no sobrescribe mlp.pkl)')

=== MLP v2 (hiperparámetros seleccionados con validación) ===
Hiperparámetros      : {'learning_rate_init': 0.001, 'hidden_layer_sizes': (64, 32), 'alpha': 0.0001}
Iteraciones          : 35
Accuracy train       : 0.6370
Accuracy test        : 0.6398
F1 weighted (test)   : 0.6780
ROC-AUC (test)       : 0.7131
Precision Deceased   : 0.3046
Recall Deceased      : 0.6907
Overfitting (train-test acc) : -0.0028

✓ Modelo guardado en data/models/mlp_v2.pkl (no sobrescribe mlp.pkl)


## 8. Búsqueda de hiperparámetros con f1_macro (v3)

Misma rejilla y mismas 25 combinaciones (`random_state=42`) que en la sección 6, pero seleccionando por `f1_macro` en vez de `f1_weighted`, y registrando también el Recall de "Deceased" (Fallecido) por candidato en `X_val`.

In [13]:
# -- Búsqueda con f1_macro (mismo espacio de 25 combinaciones); registra también Recall de Deceased por candidato ----------------------------------------------
sampler_mlp_macro = list(ParameterSampler(param_grid_mlp, n_iter=25, random_state=42))

resultados_busqueda_mlp_macro = []
for params in sampler_mlp_macro:
    modelo_tmp = MLPClassifier(
        **params, activation='relu', solver='adam', max_iter=100,
        early_stopping=True, validation_fraction=0.1, random_state=42
    )
    modelo_tmp.fit(X_train_final_sc, y_train_final, sample_weight=sample_weights_final)
    pred_val = modelo_tmp.predict(X_val_sc)
    f1_macro_val = f1_score(y_val, pred_val, average='macro')
    rec_deceased_val = recall_score((y_val == 1).astype(int), (pred_val == 1).astype(int))
    resultados_busqueda_mlp_macro.append({**params, 'f1_macro_val': f1_macro_val, 'recall_deceased_val': rec_deceased_val, 'n_iter': modelo_tmp.n_iter_})

df_busqueda_mlp_macro = pd.DataFrame(resultados_busqueda_mlp_macro).sort_values('f1_macro_val', ascending=False).reset_index(drop=True)
print(df_busqueda_mlp_macro.to_string())

mejor_mlp_macro = max(resultados_busqueda_mlp_macro, key=lambda d: d['f1_macro_val'])
mejores_params_mlp_v3 = {k: v for k, v in mejor_mlp_macro.items() if k not in ('f1_macro_val', 'recall_deceased_val', 'n_iter')}
print(f'\nMejor F1-macro (validación): {mejor_mlp_macro["f1_macro_val"]:.4f}  |  Recall Deceased (val): {mejor_mlp_macro["recall_deceased_val"]:.4f}')
print(f'Mejores hiperparámetros v3 (MLP): {mejores_params_mlp_v3}')

    learning_rate_init hidden_layer_sizes   alpha  f1_macro_val  recall_deceased_val  n_iter
0               0.0010           (64, 32)  0.0001      0.595850             0.594614      39
1               0.0050          (100, 50)  0.0100      0.593979             0.625182      31
2               0.0050          (100, 50)  0.0001      0.589475             0.635007      44
3               0.0005          (128, 64)  0.0010      0.588730             0.631732      51
4               0.0010           (64, 32)  0.0010      0.587569             0.625546      34
5               0.0005           (64, 32)  0.0010      0.586874             0.627729      47
6               0.0005           (64, 32)  0.0001      0.586375             0.627365      47
7               0.0005           (64, 32)  0.0100      0.586058             0.623362      46
8               0.0050          (128, 64)  0.0001      0.580493             0.647744      19
9               0.0050           (64, 32)  0.0001      0.579176       

## 9. Reentrenamiento final v3 (train_final + val) y evaluación en test

In [14]:
# -- Reentrenar v3 UNA sola vez con los hiperparámetros seleccionados por f1_macro, evaluar UNA sola vez en X_test ----------------------------------------------
scaler_v3 = StandardScaler()
X_train_sc_v3 = scaler_v3.fit_transform(X_train)
X_test_sc_v3  = scaler_v3.transform(X_test)
sample_weights_v3 = compute_sample_weight(class_weight='balanced', y=y_train)

mlp_v3 = MLPClassifier(
    **mejores_params_mlp_v3, activation='relu', solver='adam', max_iter=100,
    early_stopping=True, validation_fraction=0.1, random_state=42
)
mlp_v3.fit(X_train_sc_v3, y_train, sample_weight=sample_weights_v3)

pred_train_v3 = mlp_v3.predict(X_train_sc_v3)
pred_test_v3  = mlp_v3.predict(X_test_sc_v3)
proba_test_v3 = mlp_v3.predict_proba(X_test_sc_v3)[:, list(mlp_v3.classes_).index(1)]
y_test_bin_v3 = (y_test == 1).astype(int)

acc_train_v3 = accuracy_score(y_train, pred_train_v3)
acc_test_v3  = accuracy_score(y_test, pred_test_v3)
f1w_test_v3  = f1_score(y_test, pred_test_v3, average='weighted')
f1m_test_v3  = f1_score(y_test, pred_test_v3, average='macro')
auc_test_v3  = roc_auc_score(y_test_bin_v3, proba_test_v3)
prec_fall_v3 = precision_score(y_test_bin_v3, (pred_test_v3 == 1).astype(int))
rec_fall_v3  = recall_score(y_test_bin_v3, (pred_test_v3 == 1).astype(int))
overfit_v3   = acc_train_v3 - acc_test_v3

print('=== MLP v3 (hiperparámetros seleccionados con f1_macro) ===')
print(f'Hiperparámetros      : {mejores_params_mlp_v3}')
print(f'Iteraciones          : {mlp_v3.n_iter_}')
print(f'Accuracy train       : {acc_train_v3:.4f}')
print(f'Accuracy test        : {acc_test_v3:.4f}')
print(f'F1 weighted (test)   : {f1w_test_v3:.4f}')
print(f'F1 macro (test)      : {f1m_test_v3:.4f}')
print(f'ROC-AUC (test)       : {auc_test_v3:.4f}')
print(f'Precision Deceased   : {prec_fall_v3:.4f}')
print(f'Recall Deceased      : {rec_fall_v3:.4f}')
print(f'Overfitting (train-test acc) : {overfit_v3:.4f}')

joblib.dump(mlp_v3, '../data/models/mlp_v3.pkl')
joblib.dump(scaler_v3, '../data/models/scaler_mlp_v3.pkl')
print('\n✓ Modelo guardado en data/models/mlp_v3.pkl')

=== MLP v3 (hiperparámetros seleccionados con f1_macro) ===
Hiperparámetros      : {'learning_rate_init': 0.001, 'hidden_layer_sizes': (64, 32), 'alpha': 0.0001}
Iteraciones          : 35
Accuracy train       : 0.6370
Accuracy test        : 0.6398
F1 weighted (test)   : 0.6780
F1 macro (test)      : 0.5805
ROC-AUC (test)       : 0.7131
Precision Deceased   : 0.3046
Recall Deceased      : 0.6907
Overfitting (train-test acc) : -0.0028

✓ Modelo guardado en data/models/mlp_v3.pkl
